<a href="https://colab.research.google.com/github/BWMIN-Hub/SR_practice/blob/main/notebooks/19_scene_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# One scene, end to end

Pages 13 - 18 took the steps one at a time.  Here they run in order on one THEOS-2
scene, and the run ends with a report that someone else can read without the notebook.

| step | what happens |
|---|---|
| 1  prepare | read the header, convert the scene to 8 bits |
| 2  infer + write | one block at a time: read, convert, infer in tiles, convert back, write |
| 3  QC | header, radiometry, seams — each check pass or CHECK |
| 4  IQA | no-reference scores against bicubic, and a consistency check against the input |
| 5  report | one HTML file and one JSON with everything above |

The scene is 1400 x 1400 px over Bangkok: four bands, 12-bit in `uint16`, 0.5 m.
The network is the HAT x3 from page 07.

In [ ]:
import sys, urllib.request

BASE = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main'
!pip install -q timm einops
for f in ('lib/infer.py', 'lib/iqa.py', 'lib/hat_arch.py', 'lib/hat_models.py', 'lib/qcpipe.py'):
    urllib.request.urlretrieve(f'{BASE}/{f}', f.split('/')[-1])
for m in ('infer', 'iqa', 'hat_arch', 'hat_models', 'qcpipe'):
    sys.modules.pop(m, None)

import os
from IPython.display import HTML
from infer import Config
from qcpipe import *
from hat_models import load_hat

get = lambda p, out: urllib.request.urlretrieve(f'{BASE}/{p}', out)[0]
PATH = get('dataset/pipeline/scene.tif', 'scene.tif')
REF = get('dataset/pipeline/reference_hist.npz', 'reference_hist.npz')
WEIGHT = get('models/07_hat_x3/checkpoints/hat_x3_v4.pth', 'hat_x3.pth')
os.makedirs('iqa', exist_ok=True)
for f in ('niqe_model.npz', 'brisque_svr.npz'):
    get(f'results/iqa/{f}', f'iqa/{f}')

## 1. Prepare

The model takes 8-bit input, so the 12-bit values are converted to 0 - 255 and
converted back after inference.  Each band is **histogram-matched** to the 8-bit
inputs the model was trained on (`reference_hist.npz`): a value at the p-th percentile
of the scene becomes the p-th percentile of the reference.  The way back is the same
table read in reverse.

One table is used for the whole scene, so all blocks come out with the same brightness.

In [ ]:
s = scan(PATH, REF, bands=(1, 2, 3))
show_scan(s)

Near infrared is left out: the network takes three channels.  The output will have
three bands, and the checks below expect three.

## 2. Tiled inference, windowed write

Two levels of cutting.

| level | size | why |
|---|---|---|
| block | 768 px | what is read from disk and written back — bounds the **RAM** |
| tile | 192 px | what goes through the network — bounds the **GPU memory** (page 14) |

Each tile reads 32 px around itself, so 256 px go through HAT — a multiple of its 16 px
attention window.  Each block is read with the same margin, so block edges get real
neighbours just as tile edges do, and the tiles start at the block corner, so every tile
boundary in the scene lands on a multiple of 192.

The output file is created once, empty, at its final size; each block is written
straight into its window.

In [ ]:
BLOCK = 768                       # a multiple of the tile
cfg = Config(scale=3, tile=192, overlap=32, batch=4, precision='fp16')
net = load_hat(WEIGHT, scale=3)
show_blocks(s, BLOCK, cfg)

In [ ]:
st = run(PATH, 'sr.tif', net, cfg, s, block=BLOCK, model='HAT x3', value_range=1.0)

`value_range=1.0` because HAT takes 0 - 1, not 0 - 255.

The file also records how it was made — model, scale, source and conversion are
GeoTIFF tags.

In [ ]:
!gdalinfo sr.tif | grep -E 'Size is|SR_|Band [0-9]'

## 3. QC — header, radiometry, seams

Every check is a row with what was expected and what came out.

| group | asks |
|---|---|
| header | size x3, three bands, same data type and nodata, tiled and compressed, processing tags present |
| radiometry | mean within 2 %, std x 0.9 - 1.3, values in range, histogram shape, no new nodata |
| seams | the step across tile and block boundaries, against the step a few rows away |

All statistics are accumulated strip by strip.

In [ ]:
checks = qc(PATH, 'sr.tif', s, cfg, block=BLOCK)
show_qc(checks)

22 of 24 pass.

The radiometry rows pass with room to spare: the mean is within 0.3 %, the standard
deviation at 0.97 - 0.99 of the input, the histogram shape above 0.99.  Converting
to 8 bits and back kept the scene's brightness and contrast.

The range row passes, but the output stops around 1 000 - 1 200 while the input reaches
3 400 - 4 095.  The brightest pixels share the top 8-bit level, so they all come back
as a single value.

The two seam rows fail by a small margin: the step across a tile boundary is 1.15 times
the step a few rows away, and across a block boundary 1.18 times.

## 4. IQA

There is no reference image at 0.17 m for this scene, so PSNR and SSIM against a ground
truth are not possible.  Two things are:

- **no-reference scores** — NIQE, BRISQUE, PIQE on SR and on bicubic, lower is better.
  They are calibrated on this project's IKONOS images (page 11), so only the comparison
  between the two means anything.
- **consistency** — shrink the result back to 0.5 m and compare it with the input.  A
  result that changed the scene, rather than only adding detail, does not shrink back
  to what went in.

Three crops are picked automatically: the busiest, a typical and the flattest place.

In [ ]:
res = iqa_scan(PATH, 'sr.tif', s, cfg, iqa_dir='iqa')
checks += iqa_rows(res)
show_iqa(res)

Shrunk back to 0.5 m, bicubic matches the input at 42 - 45 dB.  SR matches it at
24 - 28 dB, so the result does more than add detail.  The crops show what: dark and
orange outlines along roof edges, and a painted texture that is not in the input.

The no-reference scores do not agree with each other.  NIQE prefers SR on all three
crops, BRISQUE on two (the third is a tie), PIQE on none.  Without a reference they can
only say "different", which is why the consistency number is the one that becomes a
check.

## 5. The report

One call writes `qc_report.html` — self-contained, every figure embedded — and
`qc_report.json` with the same numbers for a script to read.  The verdict at the top is
PASS only if every check passed.

In [ ]:
html = write_report('qc_report.html', PATH, 'sr.tif', s, st, checks, res)
HTML(html)

The verdict is CHECK, 5 of 27: the two seam rows and the three consistency rows.  The
header and the radiometry pass, so the file is correct and the brightness is preserved.
What fails is what the model drew: edges that do not shrink back to the input.  Whoever
decides whether this file goes out sees that on one page, without opening a notebook.

In Colab the files are in the panel on the left;
`from google.colab import files; files.download('qc_report.html')` saves one.

## 6. When the output is wrong

A copy of the result as it would be if the network's 0 - 255 output had been written
without converting back.  The report only earns its place if it catches this.

In [ ]:
show_qc(qc(PATH, break_output('sr.tif', 'bad.tif', s), s, cfg, block=BLOCK), only_bad=True)

Every radiometry row fails: the mean drops by 72 - 82 %, the values stop at 255, the
histogram shape falls to 0.2 - 0.3, and 520 000 pixels that left the network as 0 now
read as nodata.  Opened on its own with an automatic stretch, this file looks fine.